In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass



In [ ]:
cd ..

In [ ]:
import json
from pathlib import Path
import pandas as pd
p=Path(os.getenv("LEGIFRANCE_IN_DIR", "./data/in/legifrance"))
# find first excel file
files=list(p.glob('**/*'))
files=[f for f in files if f.is_file()]
print('found', len(files), 'files')
for f in files:
    print(f)
# try find xlsx
xlsx=[f for f in files if f.suffix.lower() in ('.xlsx','.xls')]
print('xlsx candidates', xlsx)
if not xlsx:
    # fallback: print first file content
    print('No xlsx found')
else:
    df=pd.read_excel(xlsx[0],engine='openpyxl')
    code_article=df.iloc[:,1].dropna().astype(str).tolist()
    print(json.dumps(code_article, ensure_ascii=False))

In [ ]:
import os, json, requests, re
from datetime import datetime, timezone
from collections.abc import Mapping, Sequence

# --- Identifiants PISTE (renseigne-les avant exécution) ---
CLIENT_ID = os.getenv("LEGIFRANCE_CLIENT_ID", "")
CLIENT_SECRET = os.getenv("LEGIFRANCE_CLIENT_SECRET", "")

ENTREE_UTILISATEUR = os.getenv("LEGIFRANCE_ENTRY", "R.331-7")
NOM_CODE = os.getenv("LEGIFRANCE_NOM_CODE", "Code général de la fonction publique")
DATE_VERSION = os.getenv("LEGIFRANCE_DATE_VERSION") or None

TOKEN_URL = os.getenv("LEGIFRANCE_TOKEN_URL", "https://oauth.piste.gouv.fr/api/oauth/token")
BASE_URL  = os.getenv("LEGIFRANCE_BASE_URL", "https://api.piste.gouv.fr/dila/legifrance/lf-engine-app")

def to_epoch_millis(date_str):
    dt = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)

# Normalisation NUM_ARTICLE: retirer les points/espaces (ex.: "R.331-13" -> "R331-13")
NUM_EXACT = re.sub(r"[.\s]", "", ENTREE_UTILISATEUR)

# 1) OAuth2
resp = requests.post(TOKEN_URL, data={
    "grant_type":"client_credentials",
    "client_id":CLIENT_ID,
    "client_secret":CLIENT_SECRET,
    "scope":"openid",
}, timeout=20)
resp.raise_for_status()
token = resp.json()["access_token"]
headers = {"Authorization": f"Bearer {token}", "accept":"application/json", "Content-Type":"application/json"}

def search(mode_typeRecherche, valeur):
    filtres = [{"facette":"NOM_CODE", "valeurs":[NOM_CODE]}]
    if DATE_VERSION:
        filtres.append({"facette":"DATE_VERSION", "singleDate": to_epoch_millis(DATE_VERSION)})

    payload = {
        "recherche": {
            "champs": [{
                "typeChamp":"NUM_ARTICLE",
                "criteres":[{"typeRecherche": mode_typeRecherche, "valeur": valeur, "operateur":"ET"}],
                "operateur":"ET"
            }],
            "filtres": filtres,
            "pageNumber": 1,
            "pageSize": 5,
            "operateur":"ET",
            "sort":"PERTINENCE",
            "typePagination":"ARTICLE"
        },
        "fond":"CODE_DATE"
    }
    r = requests.post(BASE_URL + "/search", headers=headers, data=json.dumps(payload), timeout=30)
    r.raise_for_status()
    data = r.json()
    # différentes clés possibles selon l'API
    for key in ("results","resultsList","contenus","items"):
        if isinstance(data.get(key), list) and data[key]:
            return data[key]
    if isinstance(data.get("page",{}).get("list"), list) and data["page"]["list"]:
        return data["page"]["list"]
    return []

def find_legiarti_id(obj):
    """
    Parcours récursif d'un dict/list pour trouver une chaîne qui ressemble à un ID d'article Légifrance: 'LEGIARTI...'
    """
    if isinstance(obj, str):
        if re.match(r"^LEGIARTI\d+$", obj):
            return obj
    elif isinstance(obj, Mapping):
        # essais directs usuels
        for k in ("id", "cid", "idArticle", "idContenu"):
            v = obj.get(k)
            rid = find_legiarti_id(v)
            if rid:
                return rid
        # sinon, on balaye toutes les valeurs
        for v in obj.values():
            rid = find_legiarti_id(v)
            if rid:
                return rid
    elif isinstance(obj, Sequence) and not isinstance(obj, (str, bytes)):
        for v in obj:
            rid = find_legiarti_id(v)
            if rid:
                return rid
    return None

def get_first_item_and_id(items):
    """
    Retourne (item, idTrouvé, numeroAffiché) en essayant plusieurs chemins.
    """
    if not items:
        return None, None, None
    it = items[0]
    # chemins simples
    legiarti = it.get("id") or it.get("cid") or it.get("article", {}).get("id")
    num_aff  = it.get("num") or it.get("numArticle") or it.get("article", {}).get("num")
    if not legiarti:
        legiarti = find_legiarti_id(it)
    return it, legiarti, num_aff

# 2) Recherche EXACTE sur NUM_EXACT, puis fallback COMMENCE_PAR si vide
items = search("EXACTE", NUM_EXACT)
if not items:
    # fallback: on essaie avec le préfixe "R331-13" (tel quel) puis "R331-"
    items = search("COMMENCE_PAR", NUM_EXACT) or search("COMMENCE_PAR", NUM_EXACT.rsplit("-", 1)[0] + "-")

if not items:
    raise ValueError(f"Aucun résultat pour {ENTREE_UTILISATEUR} ({NUM_EXACT}) dans « {NOM_CODE} »")

it, legiarti_id, num_aff = get_first_item_and_id(items)

if not legiarti_id:
    # Debug: montrer les clés dispo pour comprendre la structure exacte
    print("⚠️ Impossible d'extraire directement l'ID article. Aperçu du premier item :")
    # on évite d'inonder : on affiche les clés de premier niveau
    print("Clés niveau 1:", list(it.keys()))
    # et si un sous-objet 'article' existe, ses clés
    if isinstance(it.get("article"), dict):
        print("Clés it['article']:", list(it["article"].keys()))
    # on ré-essaie une dernière fois en affichant le JSON tronqué (optionnel)
    # print(json.dumps(it, ensure_ascii=False, indent=2)[:2000])
    raise ValueError("Réponse inattendue: impossible d'extraire l'ID article (pattern LEGIARTI…).")

# 3) Récupération du texte
art = requests.post(
    BASE_URL + "/consult/getArticle",
    headers=headers,
    data=json.dumps({"id": legiarti_id}),
    timeout=30
).json()

texte = (
    art.get("article", {}).get("texte")
    or art.get("texte")
    or art.get("content")
    or ""
)
num_aff = num_aff or ENTREE_UTILISATEUR

print(f"{NOM_CODE} — Article {num_aff} (ID: {legiarti_id})")
print("="*80)
print(" ".join(str(texte).split()))

In [ ]:
# Insert: loop over code_article to fetch article texts and build dict
# This cell uses the helper functions defined earlier in the notebook (search, get_first_item_and_id)
legi_results = {}
for item in code_article:
    try:
        # normalize item to match earlier NUM_EXACT behavior
        val = str(item).strip()
        num_exact = re.sub(r"[.\s]", "", val)
        items = search("EXACTE", num_exact)
        if not items:
            items = search("COMMENCE_PAR", num_exact) or []
        it, legiarti_id, num_aff = get_first_item_and_id(items)
        if not legiarti_id:
            legi_results[val] = {"error": "id_not_found", "num_aff": num_aff, "raw_item": it}
            continue
        art = requests.post(BASE_URL + "/consult/getArticle", headers=headers, data=json.dumps({"id": legiarti_id}), timeout=30).json()
        texte = (
            art.get("article", {}).get("texte")
            or art.get("texte")
            or art.get("content")
            or ""
        )
        legi_results[val] = {"id": legiarti_id, "num_aff": num_aff or val, "texte": " ".join(str(texte).split())}
    except Exception as e:
        legi_results[val] = {"error": str(e)}

# After running this cell, the `legi_results` dict maps the original input to the fetched article data.

In [ ]:
legi_results

In [ ]:
# Insert: write legi_results to out/legifrance/legi_results.json
import json
from pathlib import Path
out_dir = Path(os.getenv("LEGIFRANCE_OUT_DIR", "./data/out/legifrance"))
out_dir.mkdir(parents=True, exist_ok=True)
with open(out_dir / 'legi_results.json', 'w', encoding='utf-8') as f:
    json.dump(legi_results, f, ensure_ascii=False, indent=2)
print(f'Wrote {len(legi_results)} entries to {out_dir / "legi_results.json"}')